# M1 Notebook 24 — Numerical Linear Algebra

**Status:** Runnable first edition

## Learning objectives

- Compare QR methods.
- Use Cholesky and conjugate gradients.
- Diagnose conditioning and residuals.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_math.numerics import (
    cholesky_factor,conjugate_gradient,householder_qr,iterative_refinement,
    modified_gram_schmidt,
)


## QR factorization

In [ ]:
A=np.array([[1.,1.],[1.,1.000001],[1.,2.]])
rows=[]
for name,method in [("MGS",modified_gram_schmidt),("Householder",householder_qr)]:
    Q,R=method(A)
    rows.append({
        "method":name,
        "orthogonality_error":np.linalg.norm(Q.T@Q-np.eye(Q.shape[1])),
        "reconstruction_error":np.linalg.norm(Q@R-A),
    })
pd.DataFrame(rows)


## Least squares with QR

In [ ]:
b=np.array([1.,1.1,2.])
Q,R=householder_qr(A)
Qthin=Q[:,:A.shape[1]]
Rthin=R[:A.shape[1],:]
beta=np.linalg.solve(Rthin,Qthin.T@b)
{"beta":beta,"residual_norm":np.linalg.norm(A@beta-b)}


## Cholesky factorization

In [ ]:
S=np.array([[4.,1.],[1.,3.]])
L=cholesky_factor(S)
assert np.allclose(L@L.T,S)
L


## Conjugate gradient

In [ ]:
n=100
diag=2*np.ones(n); off=-1*np.ones(n-1)
M=np.diag(diag)+np.diag(off,1)+np.diag(off,-1)
rhs=np.ones(n)
x,h,it=conjugate_gradient(M,rhs,tol=1e-10)
{"iterations":it,"residual":np.linalg.norm(M@x-rhs)}


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.semilogy(h)
ax.set_xlabel("Iteration"); ax.set_ylabel("Residual norm")
ax.set_title("Conjugate Gradient Convergence")
plt.show()


## Conditioning

In [ ]:
matrices={
    "Identity":np.eye(5),
    "Hilbert_like":1/(np.arange(1,6)[:,None]+np.arange(1,6)[None,:]-1),
}
pd.Series({name:np.linalg.cond(M) for name,M in matrices.items()},name="condition_number")


## Iterative refinement

In [ ]:
H=matrices["Hilbert_like"]
truth=np.ones(5)
rhs=H@truth
x0=np.linalg.solve(H.astype(np.float32),rhs.astype(np.float32)).astype(float)
xr,history=iterative_refinement(H,rhs,x0,iterations=5)
{"initial_error":np.linalg.norm(x0-truth),"refined_error":np.linalg.norm(xr-truth),"residual_history":history}


## Decision Intelligence case

Solve a sparse-like network balancing system with an iterative solver.

In [ ]:
network=np.array([[3,-1,0],[-1,3,-1],[0,-1,2]],dtype=float)
demand=np.array([10,5,3],dtype=float)
allocation,h,it=conjugate_gradient(network,demand)
pd.Series(allocation,index=["Zone_A","Zone_B","Zone_C"])


## Key insight

Numerical linear algebra is about accurate, stable computation—not merely algebraic formulas.